NB.01: Wahlkampf mit Naive Bayes
	Training:
		Hypothesen H = {O, M}

		A-priori-Wahrscheinlichkeiten:
			P(O) = 4/7
			P(M) = 3/7

		Merkmal Alter:
			P(>=35 | O)	= 1/2
			P(<35 | O)	= 1/2
			P(>=35 | M) = 2/3
			P(<35 |	M)	= 1/3
		Merkmal Einkommen:
			P(hoch | O)		= 3/4
			P(niedrig | O)	= 1/4
			P(hoch | M)		= 1/3
			P(niedrig | M)	= 2/3
		Merkmal Bildung:
			P(Abitur | O)	= 1/4
			P(Bachelor | O)	= 1/4
			P(Master | O)	= 2/4
			P(Abitur | M)	= 2/3
			P(Bachelor | M)	= 1/3
			P(Master | M)	= 0/3

	Klassifikiation: <35, niedrig, Bachelor
		h = O : P(O) * P(<35 | O) * P(niedrig | O) * P(Bachelor | O) = 4/7 * 1/2 * 1/4 * 1/4 = 1/56 = 0.0178
		h = M : P(M) * P(<35 | M) * P(niedrig | M) * P(Bachelor | M) = 3/7 * 1/3 * 2/3 * 1/3 = 2/63 = 0.0317

		Der Klassifikator berechnet für jede Klasse die Wahrscheinlichkeit, dass sie unter den gegebenen Daten eintritt und wählt die Klasse mit der höchsten Wahrscheinlichkeit aus. Hier ist das Kandidat M.

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

## Using Bernoulli NB, so only checkin if a word occurs in a text, not how many times

## Processing the Datasets

# Words that don't convey much information
stopwords = {
    "the", "and", "is", "to", "of", "in", "a", "for", "on"
}

def clean_text(text):
    text = text.lower()     # To lowercase for consistency
    text = re.sub(r'[^\w\s]', '', text) # remove numbers, punctuation
    text = text.split()   # Split into individual words on spaces
    text = [t for t in text if t not in stopwords]    # Ignore stopwords

    return text # returns a List[str]

# Build the feature matrix of word x text (What words occur in which texts)
def setup_feature_matrix(word_bag, processed_texts):
    word_to_index = {word: i for i, word in enumerate(word_bag)}
    feature_matrix = np.zeros((len(processed_texts), len(word_bag)), dtype=int)
    for i, text in enumerate(processed_texts):
        for word in text:
            if word in word_to_index:
                feature_matrix[i, word_to_index[word]] = 1
    return feature_matrix

## Classificator "training"
def training(texts, labels):
    processed_texts = [set(clean_text(t)) for t in texts]

    # Build a bag of words using the processed texts
    word_bag = set()
    for text in processed_texts:
        word_bag.update(text)
    word_bag = sorted(word_bag)
    print(f"Vocabulary has {len(word_bag)} Words")

    feature_matrix = setup_feature_matrix(word_bag, processed_texts)

    # count occurence of words for ham and spam
    y = np.array(labels)
    word_counts_ham  = feature_matrix[y == 'ham'].sum(axis=0)
    word_counts_spam = feature_matrix[y == 'spam'].sum(axis=0)

    # count total ham and spam entries
    total_ham  = np.sum(y == 'ham')
    total_spam = np.sum(y == 'spam')

    # calculate a-priori probabilites : P(A)
    P_ham = total_ham / len(y)
    P_spam = total_spam / len(y)

    # calculate likelihoods for each word, given spam or ham (using laplace smoothing) : P(B | A)
    alpha = 1   # chosen arbitrarily
    P_word_if_ham  = (word_counts_ham + alpha ) / (total_ham + 2 * alpha)
    P_word_if_spam = (word_counts_spam + alpha) / (total_spam + 2 * alpha)

    return P_ham, P_spam, P_word_if_ham, P_word_if_spam, word_bag

## Classificator prediction (posterior probability)
def predicting(P_ham, P_spam, P_word_if_ham, P_word_if_spam, word_bag, texts):
    # Setup a feature matrix for test texts using the word bag of the training dataset
    processed_texts = [set(clean_text(t)) for t in texts]
    feature_matrix = setup_feature_matrix(word_bag, processed_texts)

    # use logarithmic values to avoid underflow
    log_P_word_if_ham = np.log(P_word_if_ham)
    log_1_minus_P_word_if_ham = np.log(1 - P_word_if_ham)
    log_P_word_if_spam = np.log(P_word_if_spam)
    log_1_minus_P_word_if_spam = np.log(1 - P_word_if_spam)

    # calculate logarithmic probabilities for ham/spam for all texts
    predicted_prob_ham = np.log(P_ham) + feature_matrix @ log_P_word_if_ham + (1 - feature_matrix) @ log_1_minus_P_word_if_ham
    predicted_prob_spam = np.log(P_spam) + feature_matrix @ log_P_word_if_spam + (1 - feature_matrix) @ log_1_minus_P_word_if_spam

    # compare probabilites to make predictions
    predicted_labels = np.where(predicted_prob_ham > predicted_prob_spam, 'ham', 'spam')

    return predicted_labels

## Apply training and Testing on the dataset
df = pd.read_csv("spam_ham_dataset.csv")
texts = df['text']
labels = df['label']
texts_train, texts_test, labels_train, labels_test = train_test_split(texts, labels, test_size=0.5, random_state=42)

# Train and Test the nb
P_ham, P_spam, P_word_if_ham, P_word_if_spam, word_bag = training(texts_train, labels_train)
predicted_labels = predicting(P_ham, P_spam, P_word_if_ham, P_word_if_spam, word_bag, texts_test)

# calculate accuracy by comparing predicted with true labels
accuracy = np.mean(predicted_labels == labels_test)
print(f"Accuracy: {accuracy:.2%}")

# determine most important words for spam/ham
log_ratio = np.log((P_word_if_spam) / (P_word_if_ham)) # importance calculated as the ratio of probabilites of the word occuring in spam or ham
spam_idx = np.argsort(log_ratio)[-10:][::-1]
print("top 10 spam words:")
for i in spam_idx:
    print(word_bag[i], log_ratio[i])
ham_idx = np.argsort(log_ratio)[:10]
print("\ntop 10 ham words:")
for i in ham_idx:
    print(word_bag[i], log_ratio[i])

# Combine each text in the testdata with corresponding predicted and true labels
results = pd.DataFrame({
    "text": texts_test,
    "true_label": labels_test,
    "predicted_label": predicted_labels
})
results

Vocabulary has 35537 Words
Accuracy: 82.60%
top 10 spam words:
2004 5.076919431800239
viagra 4.8174082363151545
meds 4.740447195179026
paliourg 4.678571791460938
cialis 4.612613823669141
php 4.589624305444442
pain 4.589624305444442
xp 4.566093808034248
drugs 4.566093808034248
biz 4.517303643864816

top 10 ham words:
enron -5.741057230663416
hpl -5.466841811688766
daren -5.398112479536586
meter -5.068729677885861
mmbtu -4.6970287493809035
xls -4.672834020793847
volumes -4.58322186210416
ect -4.55147316378958
sitara -4.5186833409665885
forwarded -4.516299549611312


,text,true_label,predicted_label
1566,"Subject: hpl nom for march 30 , 2001\r\n( see ...",ham,ham
1988,Subject: online pharxmacy 80 % off all meds\r\...,spam,spam
1235,Subject: re : nom / actual volume for april 17...,ham,ham
2868,Subject: re : meter 8740 dec 99\r\nrobert and ...,ham,ham
4903,Subject: re : coastal oil & gas corporation\r\...,ham,ham
...,...,...,...
4321,"Subject: opm survey help sheet\r\nhi daren ,\r...",ham,ham
2256,Subject: meter 1428 - brandywine / dupont and ...,ham,ham
313,Subject: saxet thompsonville - shut - in gas\r...,ham,ham
2281,"Subject: vastar resources , inc .\r\ngary , pr...",ham,ham
